В данном файле речь пойдет о парсинге.

С самого начала мы понимали, что готового датасета для нашего исследования не существует, поэтому выбрали парсинг данных.

Окаалось, что Wildberries не предоставляет открытого API для исследователей, но у него есть внутренние API-эндпоинты, которые использует сам сайт для загрузки каталога и карточек товаров. А на других маркетплейсах подобаться к данным было совсем тяжело, поэтому именно эти эндпоинты мы и применили.

Поиск товаров мы осуществляли через внутренний эндпоинт exactmatch, который возвращает JSON с карточками товаров — по 30 штук на страницу. Для каждой из  категорий мы собирали несколько страниц выдачи, что дало нам разнообразие брендов и цен. Но API отдаёт только базовые поля: название, бренд, цену, рейтинг, количество отзывов, размеры и цвета. Описания, состав ткани, страна производства, уход, отзывы и AI-сводки — всего этого в простом HTML ответе нет.
Страница товара на Wildberries — это SPA (Single Page Application). Контент загружается через JavaScript, и обычный HTTP-запрос возвращает почти пустой HTML. Мы перепробовали несколько подходов: искали скрытые API-эндпоинты для карточек товаров, пытались парсить HTML через BeautifulSoup, тестировали прямые запросы по ссылке. Ни один из них не давал описания. В итоге единственным работающим решением был Selenium — настоящий браузер, который открывает страницу товара, кликает на кнопку «Характеристики и описание» и забирает появившийся текст. Это крайне медленно — около 5-7 секунд на товар, но только так можно было получить все основные текстовые данные.
С отзывами использовалась та же история, но здесь оказалось еще сложнее: кнопка «Смотреть все отзывы» в разных версиях сайта имела разные селекторы. В целом, WB часто меняет эндпоинты - видимо, своя защита от парсинга. Из-за этого часть данных всё же не была получена, поэтому пришлось работать с пропусками.

Более того, Wildberries требует авторизационные куки для доступа к внутреннему API, поэтому токен надо было также доставать.

Поскольку на 3708 товаров с Selenium требовалось около 5-7 часов непрерывной работы, мы распараллелили сбор на 4 процесса, каждый обрабатывал свою часть категорий. Это сократило время, но из-за ограничений процессы иногда вставали — приходилось добавлять паузы и перезапускать.

Сначала был запущен тестовый парсер:

In [1]:
import requests
import time
import random
import pandas as pd
import json
import os
import re
from datetime import datetime
from fake_useragent import UserAgent
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


CATEGORIES = ["платье вечернее"]
PAGES_PER_CATEGORY = 1
PRODUCTS_PER_PAGE = 5
MIN_DELAY = 3.0
MAX_DELAY = 5.0
MAX_RETRIES = 3
PRODUCTS_FILE = "wb_test_5.csv"
os.makedirs("wb_backups", exist_ok=True)


chrome_options = Options()
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--disable-blink-features=AutomationControlled')
chrome_options.add_argument('--disable-gpu')
chrome_options.add_argument('--window-size=1920,1080')
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option('useAutomationExtension', False)
chrome_options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36')


class WildberriesParser:

    def __init__(self):
        self.ua = UserAgent()
        self.search_url = "https://www.wildberries.ru/__internal/u-search/exactmatch/ru/common/v18/search"
        self.session = requests.Session()
        self.cookies = {
            '_wbauid': '1542039911763969751',
            'x_wbaas_token': '1.1000.d33dffe5b1194bc39776f55e96429aee.MHwzNy43Ni4xNTkuNTR8TW96aWxsYS81LjAgKFdpbmRvd3MgTlQgMTAuMDsgV2luNjQ7IHg2NCkgQXBwbGVXZWJLaXQvNTM3LjM2IChLSFRNTCwgbGlrZSBHZWNrbykgQ2hyb21lLzE0Ni4wLjAuMCBZYUJyb3dzZXIvMjYuNC4wLjAgU2FmYXJpLzUzNy4zNnwxNzgxNjAyMDA0fHJldXNhYmxlfDJ8ZXlKb1lYTm9Jam9pSW4wPXwwfDN8MTc4MTQ3MjQwNHwx.MEUCIDjN/l984esqOqBPoFqGY8V1rWehkKQOlD10ALrWUgOPAiEA7zor0yC8Uy+oSOOeJokZRZ871ynkjbd1Lcw94p7GKiQ=',
            '_cp': '1',
        }
        self.x_spa_version = '14.13.6'
        self.device_id = 'site_2633732a97854ab2bcdbae007471feb4'
        self.driver = webdriver.Chrome(options=chrome_options)
        self.driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
        print("Готово")

    def _generate_query_id(self):
        return f"qid1542039911763969751{datetime.now().strftime('%Y%m%d%H%M%S')}{random.randint(100, 999)}"

    def _headers(self):
        return {
            'accept': '*/*', 'accept-language': 'ru,en;q=0.9',
            'deviceid': self.device_id, 'priority': 'u=1, i',
            'referer': 'https://www.wildberries.ru/',
            'sec-ch-ua': '"Chromium";v="146", "Not-A.Brand";v="24", "YaBrowser";v="26.4", "Yowser";v="2.5"',
            'sec-ch-ua-mobile': '?0', 'sec-ch-ua-platform': '"Windows"',
            'sec-fetch-dest': 'empty', 'sec-fetch-mode': 'cors',
            'sec-fetch-site': 'same-origin',
            'user-agent': self.ua.random,
            'x-queryid': self._generate_query_id(),
            'x-requested-with': 'XMLHttpRequest',
            'x-spa-version': self.x_spa_version,
            'x-userid': '0',
        }

    def _request(self, url, params=None):
        for attempt in range(MAX_RETRIES):
            try:
                if params:
                    resp = self.session.get(url, params=params, headers=self._headers(), cookies=self.cookies, timeout=30)
                else:
                    resp = self.session.get(url, headers=self._headers(), cookies=self.cookies, timeout=30)
                if resp.status_code == 429:
                    wait = int(resp.headers.get('Retry-After', 10))
                    print(f"  лимит запросов, жду {wait}с")
                    time.sleep(wait)
                    continue
                if resp.status_code in (404, 498):
                    return None
                resp.raise_for_status()
                return resp
            except:
                if attempt < MAX_RETRIES - 1:
                    time.sleep((attempt + 1) * 2)
        return None

    def search_page(self, query, page):
        params = {
            "ab_testing": "false", "appType": "1", "curr": "rub",
            "dest": "-1257786", "hide_dtype": "9;11;15",
            "hide_vflags": "4294967296", "inheritFilters": "false",
            "lang": "ru", "locale": "ru", "query": query,
            "resultset": "catalog", "sort": "popular", "spp": "30",
            "suppressSpellcheck": "false", "page": str(page),
            "limit": str(PRODUCTS_PER_PAGE),
        }
        resp = self._request(self.search_url, params)
        if not resp:
            return []
        try:
            data = resp.json()
        except:
            return []
        products_raw = data.get('products', [])
        results = []
        for p in products_raw:
            product_id = p.get('id')
            brand = p.get('brand', '')
            name = p.get('name', '')
            full_name = f"{brand} {name}".strip() if brand else name
            sizes = p.get('sizes', [])
            basic_price = sizes[0].get('price', {}).get('basic', 0) if sizes else 0
            product_price = sizes[0].get('price', {}).get('product', 0) if sizes else 0
            discount_pct = round((1 - product_price / basic_price) * 100) if basic_price and product_price and basic_price > product_price else 0
            sizes_names = [s.get('name', '') for s in sizes if s.get('name')]
            colors_list = p.get('colors', [])
            colors_names = [c.get('name', '') for c in colors_list if c.get('name')]
            results.append({
                'product_id': product_id,
                'imt_id': p.get('root', product_id),
                'name': full_name, 'brand': brand,
                'price': product_price / 100 if product_price else 0,
                'price_no_discount': basic_price / 100 if basic_price else 0,
                'discount_pct': discount_pct,
                'rating': p.get('reviewRating', 0) or p.get('rating', 0),
                'reviews_count': p.get('feedbacks', 0) or 0,
                'subject_id': p.get('subjectId', ''),
                'supplier_rating': p.get('supplierRating', 0),
                'sizes': ', '.join(sizes_names) if sizes_names else '',
                'colors': ', '.join(colors_names) if colors_names else '',
                'category_query': query, 'page': page,
                'collected_at': datetime.now().isoformat()
            })
        return results

    def get_all_details(self, product_id):
        url = f"https://www.wildberries.ru/catalog/{product_id}/detail.aspx"
        result = {
            'description': '',
            'composition': '', 'color': '', 'gender': '',
            'country': '', 'care': '',
            'ai_summary': '',
            'reviews_text': '',
        }
        try:
            self.driver.get(url)
            time.sleep(3)

            try:
                btn = WebDriverWait(self.driver, 10).until(
                    EC.element_to_be_clickable((By.CLASS_NAME, "btnDetailText--nrkiv"))
                )
                self.driver.execute_script("arguments[0].scrollIntoView(true);", btn)
                time.sleep(1)
                btn.click()
                time.sleep(2)
                print("    характеристики: кликнули")
            except:
                print("    характеристики: кнопка не найдена")

            try:
                desc = WebDriverWait(self.driver, 5).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "#section-description p"))
                )
                result['description'] = desc.text.strip()[:3000]
                print(f"    описание: {len(result['description'])} зн")
            except:
                print("    описание: не найдено")

            try:
                rows = self.driver.find_elements(By.CSS_SELECTOR, ".content--zb_r9 tr")
                for row in rows:
                    try:
                        th = row.find_element(By.TAG_NAME, "th").text.strip().lower()
                        td = row.find_element(By.TAG_NAME, "td").text.strip()
                        if 'состав' in th: result['composition'] = td
                        elif 'цвет' in th: result['color'] = td
                        elif 'пол' in th: result['gender'] = td
                        elif 'страна' in th: result['country'] = td
                        elif 'уход' in th: result['care'] = td
                    except:
                        pass
                print(f"    состав={result['composition'] or 'нет'}, страна={result['country'] or 'нет'}")
            except:
                print("    характеристики: не найдены")

            try:
                reviews_btn = self.driver.find_element(By.CSS_SELECTOR, "a.comments__btn-all")
                reviews_url = reviews_btn.get_attribute("href")
                self.driver.get(reviews_url)
                time.sleep(3)
                print(f"    перешли к отзывам")
            except:
                print("    кнопка отзывов не найдена")

            try:
                ai = self.driver.find_element(By.CLASS_NAME, "generative-feedback__text")
                result['ai_summary'] = ai.text.strip()
                print(f"    AI-сводка: найдена")
            except:
                print("    AI-сводка: не найдена")

            try:
                items = self.driver.find_elements(By.CSS_SELECTOR, "li.comments__item")
                texts = []
                for item in items[:5]:
                    try:
                        t = item.find_element(By.CLASS_NAME, "feedback__text")
                        if t:
                            texts.append(t.text.strip())
                    except:
                        pass
                result['reviews_text'] = ' ||| '.join(texts)
                print(f"    отзывов: {len(texts)}")
            except:
                print("    отзывы не найдены")

        except Exception as e:
            print(f"    ошибка: {e}")
        return result

    def run(self):
        all_products = []
        seen_ids = set()

        print(f"\nТест на 5 товарах")
        for query in CATEGORIES:
            print(f"\nКатегория: {query}")
            products = self.search_page(query, 1)
            if not products:
                break
            print(f"  собрано из API: {len(products)} товаров")

            for i, prod in enumerate(products):
                pid = prod['product_id']
                if pid in seen_ids:
                    continue
                seen_ids.add(pid)

                print(f"\n  [{i+1}] {prod['name'][:70]}...")
                print(f"    цена: {prod['price']} руб, рейтинг: {prod['rating']}, отзывов: {prod['reviews_count']}")

                details = self.get_all_details(pid)
                prod.update(details)

                desc_ok = 'есть' if prod['description'] else 'нет'
                ai_ok = 'есть' if prod['ai_summary'] else 'нет'
                rev_ok = 'есть' if prod['reviews_text'] else 'нет'
                print(f"    итог: описание={desc_ok}, AI={ai_ok}, отзывы={rev_ok}")

                all_products.append(prod)
                time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

        df = pd.DataFrame(all_products)
        if not df.empty:
            df = df.drop_duplicates(subset='product_id')
        df.to_csv(PRODUCTS_FILE, index=False, encoding='utf-8-sig')

        print(f"\n---")
        print(f"Товаров: {len(df)}")
        print(f"С описанием: {(df['description'] != '').sum()}")
        print(f"С AI-сводкой: {(df['ai_summary'] != '').sum()}")
        print(f"С отзывами: {(df['reviews_text'] != '').sum()}")
        print(f"Файл: {PRODUCTS_FILE}")

        self.driver.quit()
        return df


parser = WildberriesParser()
df = parser.run()

Готово

Тест на 5 товарах

Категория: платье вечернее
  собрано из API: 5 товаров

  [1] NOVOCH Стильное платье мини с шлейфом...
    цена: 1299.0 руб, рейтинг: 4.5, отзывов: 1680
    характеристики: кликнули
    описание: 810 зн
    состав=95 % полиэстр, 5% спандекс, страна=нет
    перешли к отзывам
    AI-сводка: не найдена
    отзывов: 5
    итог: описание=есть, AI=нет, отзывы=есть

  [2] ALURA store Платье вечернее корсетное с разрезом в пол...
    цена: 5497.0 руб, рейтинг: 4.9, отзывов: 289
    характеристики: кликнули
    описание: 1748 зн
    состав=шелк 90%; вискоза 10%, страна=Киргизия
    перешли к отзывам
    AI-сводка: найдена
    отзывов: 5
    итог: описание=есть, AI=есть, отзывы=есть

  [3] Miss Brand Платье праздничное вечернее длинное...
    цена: 4096.0 руб, рейтинг: 4.9, отзывов: 66
    характеристики: кликнули
    описание: 1920 зн
    состав=полиэстер, страна=Киргизия
    перешли к отзывам
    AI-сводка: найдена
    отзывов: 3
    итог: описание=есть, AI=есть, отз

Далее парсер использовал следующий код из файла Sbor_dannyh, но на этом компьютере он был остановлен: суммарно все занимало около 30-40 часов, слишком много. На другом компьютере были получены данные из 4 распараллеленных процессах - wb_products_full, сюда прикреплен уже итоговый файл.

Но там были проблемы с отзывами: они не считались из-за ошибки в парсинге, поэтому работа распараллелилась, одновременно обрабатывались чисовые и категориальные характеристики, а текстовые продолжали парситься, но чуть более легко, без API, просто по ID:

In [2]:
df = pd.read_csv("wb_products_full.csv")
total = len(df)
chunk = total // 4

for i in range(4):
    start = i * chunk
    end = start + chunk if i < 3 else total
    part = df.iloc[start:end]
    part.to_csv(f"wb_part_for_reviews_{i+1}.csv", index=False)
    print(f"Часть {i+1}: строки {start}-{end-1}, всего {len(part)} товаров")

Часть 1: строки 0-926, всего 927 товаров
Часть 2: строки 927-1853, всего 927 товаров
Часть 3: строки 1854-2780, всего 927 товаров
Часть 4: строки 2781-3707, всего 927 товаров


Далее все процессы обрабатывались аналогично этому:

In [ ]:
import pandas as pd
import time
import random
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

INPUT_FILE = "wb_part_for_reviews_1.csv"
OUTPUT_FILE = "wb_reviews_part1.csv"

chrome_options = Options()
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--disable-blink-features=AutomationControlled')
chrome_options.add_argument('--window-size=1920,1080')
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option('useAutomationExtension', False)

driver = webdriver.Chrome(options=chrome_options)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

df = pd.read_csv(INPUT_FILE)
print(f"Процесс 1: {len(df)} товаров")

results = []
ai_count = 0
rev_count = 0

for idx, row in df.iterrows():
    pid = row['product_id']
    imt = row['imt_id']
    ai_text = ''
    rev_text = ''
    
    try:
        # Загружаем страницу товара
        driver.get(f"https://www.wildberries.ru/catalog/{pid}/detail.aspx")
        time.sleep(3)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        
        # Переходим на страницу отзывов
        try:
            btn = driver.find_element(By.XPATH, "//a[contains(text(), 'Смотреть все отзывы')]")
            driver.execute_script("arguments[0].click();", btn)
        except:
            driver.get(f"https://www.wildberries.ru/catalog/{pid}/feedbacks?imtId={imt}")
        
        # Ждём появления хотя бы одного отзыва или AI-сводки
        try:
            WebDriverWait(driver, 10).until(
                lambda d: d.find_elements(By.CSS_SELECTOR, "li.comments__item") or 
                         d.find_elements(By.CLASS_NAME, "generative-feedback__text")
            )
        except:
            pass
        
        # AI-сводка
        try:
            ai_text = driver.find_element(By.CLASS_NAME, "generative-feedback__text").text.strip()
            ai_count += 1
        except:
            pass
        
        # Отзывы
        try:
            items = driver.find_elements(By.CSS_SELECTOR, "li.comments__item")
            texts = []
            for item in items[:5]:
                try:
                    t = item.find_element(By.CLASS_NAME, "feedback__text")
                    if t:
                        texts.append(t.text.strip())
                except:
                    pass
            if texts:
                rev_text = ' ||| '.join(texts)
                rev_count += 1
        except:
            pass
        
    except:
        pass
    
    results.append({'product_id': pid, 'ai_summary': ai_text, 'reviews_text': rev_text})
    
    if (idx + 1) % 10 == 0:
        print(f"  [{idx+1}/{len(df)}] AI: {ai_count}, отзывы: {rev_count}")
    
    time.sleep(random.uniform(1.5, 2.5))

driver.quit()
pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"Готово: AI={ai_count}, отзывы={rev_count}")

И соединились. Опять же, все это было в разных файлах и на разных компьютерах для скорости, поэтому итоговые данные уже сразу прикреплены в файле wb_products_full_imputed.csv.